In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")

GPU available: True
GPU name     : Tesla T4
VRAM         : 15.6 GB


In [ ]:
 !pip install faster-whisper soundfile scipy groq python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install import-ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.5 MB/s eta 0:00:00


In [ ]:
!pip install pyannote.audio

In [ ]:
import os
HF_TOKEN       = os.getenv("HF_TOKEN")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
# check token works
user = api.whoami(token=HF_TOKEN)
print(f"Logged in as: {user['name']}")

# check model accessible
model_info = api.model_info("pyannote/embedding", token=HF_TOKEN)
print(f"Model accessible: {model_info.id}")

Logged in as: snehaaiml
Model accessible: pyannote/embedding


In [ ]:
import numpy as np
import torch
from pyannote.audio import Pipeline
from pydub import AudioSegment
import os
import soundfile as sf
from faster_whisper import WhisperModel
import scipy.signal as scs

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
def load_diarization_pipeline():
    import torch
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        token=HF_TOKEN
    )
    pipeline.to(torch.device("cuda"))
    return pipeline

def convert_mp3_to_wav(inp_path,out_path):
    audio = AudioSegment.from_mp3(inp_path)
    audio.export(out_path,format="wav")

def prepare_audio(wav_path):
    audio_data, sample_rate = sf.read(wav_path, dtype="float32")
    print(f"Original sample rate: {sample_rate}Hz")
    print(f"Original shape: {audio_data.shape}")
    print(f"Original dtype: {audio_data.dtype}")

    # Step 1 — Convert to mono
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)

    # Step 2 — Resample to 16kHz
    if sample_rate != 16000:
        num_samples = int(len(audio_data) * 16000 / sample_rate)
        audio_data = scs.resample(audio_data, num_samples)  # returns numpy array
        sample_rate = 16000
        print(f"Resampled to: {sample_rate}Hz")

    # Step 3 — Ensure numpy array before converting to tensor
    audio_data = np.array(audio_data, dtype=np.float32)

    # Step 4 — Convert to tensor correctly
    waveform = torch.from_numpy(audio_data).unsqueeze(0)

    print(f"Waveform shape: {waveform.shape}")            # should be (1, N)
    print(f"Waveform dtype: {waveform.dtype}")            # should be torch.float32

    return {
        "waveform": waveform,
        "sample_rate": sample_rate
    }


def diarize_audio(pipeline, audio_input):

    diarization = pipeline(audio_input)

    annotation = diarization.speaker_diarization

    speaker_segments = []

    print("\nPYANNOTE RESULT\n")

    for turn, _, speaker in annotation.itertracks(yield_label=True):

        speaker_segments.append({
            "speaker": speaker,
            "start": turn.start,
            "end": turn.end
        })

        print(
            f"[{turn.start:.1f}s-{turn.end:.1f}s] {speaker}"
        )

    return speaker_segments


In [ ]:
def merge_whisper_first(whis_seg,pyann_seg):
    # both are list
    merged_seg=[]

    for seg in whis_seg:
        best_overlap=-1
        best_speaker=None
        best_dist=float("inf")
        nearest_speaker=None
        for p_seg in pyann_seg:
            if p_seg["start"]>seg["end"]:
                break

            overlap=min(seg["end"],p_seg["end"])-max(p_seg["start"],seg["start"])
            if overlap>best_overlap:
                best_overlap=overlap
                best_speaker=p_seg["speaker"]
            distance=min(
                abs(seg["start"]-p_seg["end"]),
                abs(seg["end"]-p_seg["start"])
            )
            if distance<best_dist:
                best_dist=distance
                nearest_speaker=p_seg["speaker"]

        # use overlap speaker if found, else nearest speaker
        final_speaker = best_speaker if best_overlap > 0 else nearest_speaker
        merged_seg.append({
            "start":seg["start"],
            "end":seg["end"],
            "speaker":final_speaker if final_speaker else "UNKNOWN",
            "text":seg["text"]
        })
    return merged_seg

In [ ]:
import json
import re
from typing import Optional


def parse_llm_summary(raw_output: str) -> dict:
    """

    Handles:
      - ```json ... ``` or ``` ... ``` code fences
      - stray text before/after the JSON block
      - trailing commas
      - smart/curly quotes
      - single-quoted keys/values (rare but happens with some model outputs)
    """
    if not raw_output or not raw_output.strip():
        raise ValueError("Empty LLM output passed to parse_llm_summary")

    text = raw_output.strip()

    # 1. Strip markdown code fences if present
    fence_match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()

    # 2. If there's still leading/trailing junk, isolate the outermost {...}
    #    (find first '{' and matching last '}')
    if not text.startswith("{"):
        first_brace = text.find("{")
        last_brace = text.rfind("}")
        if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
            text = text[first_brace:last_brace + 1]
        else:
            raise ValueError(f"No JSON object found in output:\n{raw_output[:300]}")

    # 3. First attempt: strict parse
    parsed = _try_json_loads(text)
    if parsed is not None:
        return _clean_dict(parsed)

    # 4. Fallback: repair common LLM JSON issues, then retry
    repaired = _repair_json_text(text)
    parsed = _try_json_loads(repaired)
    if parsed is not None:
        return _clean_dict(parsed)

    raise ValueError(f"Could not parse JSON even after repair attempts:\n{text[:500]}")


def _try_json_loads(text: str) -> Optional[dict]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def _repair_json_text(text: str) -> str:
    # Remove trailing commas before } or ]
    text = re.sub(r",\s*([}\]])", r"\1", text)

    # Normalize smart quotes to straight quotes
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")

    # Convert single-quoted keys/values to double-quoted (best-effort, safe-ish)
    # Only applied if double-quoted parse already failed
    text = re.sub(r"'([^']*)'\s*:", r'"\1":', text)   # keys
    text = re.sub(r":\s*'([^']*)'", r': "\1"', text)  # string values

    # Remove any stray control characters that break json.loads
    text = re.sub(r"[\x00-\x1f\x7f]", "", text)

    return text


def _clean_dict(d: dict) -> dict:
    """Light normalization that's safe across any schema."""
    cleaned = {}
    for k, v in d.items():
        key = k.strip() if isinstance(k, str) else k
        if isinstance(v, str):
            v = v.strip()
        elif isinstance(v, list):
            v = [item.strip() if isinstance(item, str) else item for item in v]
        cleaned[key] = v
    return cleaned

In [ ]:
!pip install sentence_transformers

In [ ]:
import re
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

class RoleIdentifier:

  def __init__(self,model_name,str="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    print("Intializing model...")
    self.model=SentenceTransformer(model_name)
    print("Model intialized")
    if torch.cuda.is_available():
      self.model.to(torch.device("cuda"))

    # Semantic Anchors for Dialogue Openings/Closings
    self.call_taker_concepts = [
            "आपातकालीन dial ११2 में आपका स्वागत है बताइए क्या हुआ",
            "नमस्ते डायल 112 बात कर रहे हैं",
            "आपकी जानकारी दर्ज कर ली गई है जल्द ही पुलिस सहायता पहुंचेगी धन्यवाद",
            "थाना क्षेत्र कौन सा लगता है आपका"
    ]
    self.caller_concepts = [
            "बचाओ मेरा एक्सीडेंट हो गया है मदद चाहिए",
            "मेरे खाते से पैसे कट गए हैं फ्रॉड हुआ है ऑनलाइन धोखाधड़ी",
            "यहाँ बहुत भीषण आग लग गयी है जल्दी दमकल गाडी भेजिए",
            "मुझे पुलिस की मदद चाहिए कोई झगड़ा कर रहा है"
    ]

    # Pre-compute static embeddings once at startup
    self.ref_taker_embeddings = self.model.encode(self.call_taker_concepts, convert_to_tensor=True)
    self.ref_caller_embeddings = self.model.encode(self.caller_concepts, convert_to_tensor=True)


    # Hindi question tokens for conversational structure checks
    self.hindi_question_words = ["क्या", "कहाँ", "कहा", "कौन", "कब", "कैसे", "कितना", "कितने", "किधर"]

  def is_question(self,text:str,embedding:torch.Tensor)->bool:
      if "?" in text or any(q_word in text for q_word in self.hindi_question_words):
        return True
      # semantic question matching
      sim_matrix = util.cos_sim(embedding, self.ref_taker_embeddings)
      return float(torch.max(sim_matrix)) > 0.60

  def process_transcript(self,merge_seg:list)->list:
      if not merge_seg:
        return merge_seg
      speaker_profiles={}
      all_texts=[seg["text"] for seg in merge_seg]
      print(type(all_texts))

      embeddings=self.model.encode(all_texts,convert_to_tensor=True)
      print(type(embeddings))
      print(embeddings[:2])

      # score1: context analysis focusing heavily on begg and endings
      for idx,seg in enumerate(merge_seg):
        speaker=seg["speaker"]
        if speaker not in speaker_profiles:
          speaker_profiles[speaker]={"context_score":0,"qna_score":0}
        is_edge=(idx<3 or idx>=len(merge_seg)-2)
        wt=3.0 if is_edge else 0.5

        sim_taker=float(torch.max(util.cos_sim(embeddings[idx],self.ref_taker_embeddings)))
        sim_caller=float(torch.max(util.cos_sim(embeddings[idx],self.ref_caller_embeddings)))

        speaker_profiles[speaker]["context_score"] += (sim_taker - sim_caller) * wt

      for i in range(len(merge_seg) - 1):
            current_seg = merge_seg[i]
            next_seg = merge_seg[i+1]

            # Look for explicit transitions between different human entities
            if current_seg["speaker"] != next_seg["speaker"]:
                if self.is_question(current_seg["text"], embeddings[i]):
                    # Award a conversational dominance point to the question generator
                    speaker_profiles[current_seg["speaker"]]["qna_score"] += 1.5

        # Final Evaluation: Calculate the complete composite vector mapping
      final_rankings = []
      for speaker, metrics in speaker_profiles.items():
            total_composite_score = metrics["context_score"] + metrics["qna_score"]
            final_rankings.append((speaker, total_composite_score))

        # Sort profiles descending
      final_rankings.sort(key=lambda x: x[1], reverse=True)

        # Assign explicit roles securely based on structural traits
      call_taker_raw_id = final_rankings[0][0]
      caller_raw_id = final_rankings[1][0] if len(final_rankings) > 1 else None

        # Transform and rewrite raw mapping into systemic pipeline schema
      normalized_transcript = []
      for seg in merge_seg:
            new_seg = seg.copy()
            if seg["speaker"] == call_taker_raw_id:
                new_seg["speaker"] = "CALL_TAKER"
            elif caller_raw_id and seg["speaker"] == caller_raw_id:
                new_seg["speaker"] = "CALLER"
            else:
                new_seg["speaker"] = "CALLER"
            normalized_transcript.append(new_seg)

      return normalized_transcript






In [ ]:
!pip install google-genai

In [ ]:
import os
import json
import time
import soundfile as sf
import scipy.signal as scs
import numpy as np
from faster_whisper import WhisperModel
import torch
from pyannote.audio import Pipeline
from pydub import AudioSegment
from google.colab import userdata
from datetime import datetime
# from google import genai
# from google.genai import types
import traceback
# from google.genai.errors import ServerError, APIError

whisper_model = None
diarization_pipeline = None

# gemini_client = None

def init_shared_models():
    global whisper_model, diarization_pipeline
    print("\n Initializing models...")
    if whisper_model is None:
        print("loading whisper model on CUDA...\n")
        whisper_model = WhisperModel(
                "large-v2",
                device="cuda",
                compute_type="float16"
        )

    if diarization_pipeline is None:
        print("loading pyannote pipeline...\n")
        diarization_pipeline = load_diarization_pipeline()


    # if gemini_client is None:
    #     print("loading gemini client...\n")
    #     api_key = os.getenv("GEMINI2")
    #     gemini_client = genai.Client(api_key=api_key)

pipeline_start = time.time()
timings = {}

def pipeline(wav_path):
    global whisper_model, diarization_pipeline
    if whisper_model is None or diarization_pipeline is None:
      init_shared_models()

    def log_time(stage, t_start):
      elapsed = round(time.time() - t_start, 3)
      timings[stage] = elapsed
      print(f" {stage:30s} {elapsed}s")
      return elapsed

    # step 1 : track audio specs-----------
    audio_info = sf.info(wav_path)
    total_duration = audio_info.duration
    print(f"  audio duration: {round(total_duration, 2)}s\n")

    # step 2 : prepare audio------------
    print("conveting to mono")
    t = time.time()
    audio_inp = prepare_audio(wav_path)
    waveform = audio_inp["waveform"].squeeze(0).numpy()
    log_time("audio prepare (resample)", t)

    # ── Step 3: whisper transcription ────────────────────────────────────────
    print("\n running transcription...\n")
    t = time.time()
    segments, info = whisper_model.transcribe(
            waveform,
            language="en",
            task="transcribe",
            word_timestamps=True,
            beam_size=5,
            temperature=0.0,
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            vad_filter=True,
            vad_parameters=dict(
                threshold=0.45,
                min_silence_duration_ms=500,
            ),
            compression_ratio_threshold=2.4
    )

    whis_seg = []
    print("--- WHISPER SEGMENTS OUTPUT ---")
    for seg in segments:
            seg_dict = {"start": seg.start, "end": seg.end, "text": seg.text.strip()}
            whis_seg.append(seg_dict)
            print(f"[{seg.start:.1f}s - {seg.end:.1f}s]: {seg.text.strip()}")
    print("-------------------------------\n")

    log_time("whisper transcription", t)

    # step 4 : PyAnnote Diarization--------
    print("\nrunning diarization...\n")
    t = time.time()
    pyann_seg = diarize_audio(diarization_pipeline, audio_inp)
    log_time("pyannote diarization", t)

    # step 5 : merge & align segments-------
    print("\nmerging...\n")
    t = time.time()
    merge_seg = merge_whisper_first(whis_seg, pyann_seg)
    log_time("merger + normalize", t)

    # Print Full Merged Segments Result
    print("\n--- FINAL MERGED SEGMENTS OUTPUT ---")
    for item in merge_seg:
        print(f"{item.get('speaker', 'Unknown')} [{item.get('start', 0):.1f}s - {item.get('end', 0):.1f}s]: {item.get('text', '')}")
    print("------------------------------------\n")

    audio_base_name = os.path.splitext(os.path.basename(wav_path))[0]

    # Generate a unique timestamp string (e.g., "20260729_232619")
    timestamp = datetime.now().strftime("%d%m%Y_%H%M")

    # Define the unique output path incorporating both the audio name and timestamp
    output_json_path = f"/content/drive/MyDrive/MSummarizer/{audio_base_name}_{timestamp}.json"

    complete_output_data = {
        "merged_segments": merge_seg
    }
    # Save the data into a JSON file
    with open(output_json_path, "w", encoding="utf-8") as json_file:
        json.dump(complete_output_data, json_file, indent=4, ensure_ascii=False)

    print(f"Successfully saved all transcription and diarization results to: {output_json_path}")

    return merge_seg



ModuleNotFoundError: No module named 'faster_whisper'

In [ ]:
# res=pipeline("/content/drive/MyDrive/MSummarizer/ES2002a.Mix-Headset.wav")


 Initializing models...
loading whisper model on CUDA...

loading pyannote pipeline...

  audio duration: 1272.64s

conveting to mono
Original sample rate: 16000Hz
Original shape: (20362240,)
Original dtype: float32
Waveform shape: torch.Size([1, 20362240])
Waveform dtype: torch.float32
 audio prepare (resample)       0.135s

 running transcription...

--- WHISPER SEGMENTS OUTPUT ---
[4.5s - 7.7s]: My gosh, you've already produced a PowerPoint presentation.
[7.8s - 9.3s]: I think it's already on, actually.
[15.1s - 16.2s]: God, I don't know if this is going to work.
[33.2s - 35.0s]: I've plugged it in the back, but...
[39.8s - 40.8s]: OK, right.
[48.0s - 53.8s]: OK. Right.
[56.3s - 59.1s]: Well, this is the kick-off meeting for our project.
[63.6s - 66.6s]: And this is just what we're going to be doing over the next 25 minutes.
[68.5s - 74.4s]: So first of all, just to kind of make sure that we all know each other, I'm Laura and I'm the project manager.
[74.9s - 75.1s]: Great.
[75.9s 

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)



PYANNOTE RESULT

[0.0s-0.9s] SPEAKER_03
[4.8s-7.9s] SPEAKER_02
[7.4s-9.5s] SPEAKER_03
[8.6s-9.1s] SPEAKER_02
[10.6s-10.7s] SPEAKER_02
[12.2s-12.7s] SPEAKER_03
[15.0s-16.1s] SPEAKER_03
[19.3s-20.2s] SPEAKER_00
[20.7s-21.6s] SPEAKER_03
[25.9s-27.3s] SPEAKER_01
[30.5s-30.9s] SPEAKER_03
[32.2s-32.9s] SPEAKER_00
[33.4s-35.0s] SPEAKER_03
[35.3s-35.9s] SPEAKER_00
[38.1s-39.1s] SPEAKER_00
[39.8s-42.2s] SPEAKER_03
[42.2s-42.2s] SPEAKER_03
[44.8s-46.1s] SPEAKER_00
[48.2s-49.0s] SPEAKER_00
[48.4s-48.4s] SPEAKER_03
[50.4s-51.1s] SPEAKER_03
[53.6s-54.1s] SPEAKER_03
[55.9s-77.4s] SPEAKER_03
[67.2s-67.3s] SPEAKER_00
[67.3s-67.4s] SPEAKER_02
[67.4s-67.4s] SPEAKER_00
[75.0s-75.1s] SPEAKER_02
[77.4s-80.6s] SPEAKER_00
[80.8s-81.3s] SPEAKER_03
[82.1s-84.5s] SPEAKER_02
[85.9s-88.7s] SPEAKER_01
[89.3s-101.7s] SPEAKER_03
[104.9s-106.2s] SPEAKER_03
[108.7s-132.1s] SPEAKER_03
[132.5s-139.4s] SPEAKER_00
[135.1s-135.5s] SPEAKER_03
[137.1s-137.2s] SPEAKER_02
[139.5s-139.5s] SPEAKER_00
[139.5s-142.0s] SPEAKER_02


In [ ]:
# import json

# # Define the output path in your Google Drive
# output_json_path = "/content/drive/MyDrive/MSummarizer/transcription_results.json"

# # Prepare the complete data structure containing Whisper segments, Diarization, and the final Merged results
# # Note: Ensure you have 'whis_seg', 'pyann_seg', and 'merge_seg' generated from your pipeline run.
# complete_output_data = {
#     "audio_file": audio_path,
#     "whisper_segments": whis_seg,
#     "pyannote_segments": [{"start": float(turn.start), "end": float(turn.end), "speaker": speaker} for turn, _, speaker in pyann_seg.itertracks(yield_label=True)],
#     "merged_segments": merge_seg
# }

# # Save the data into a JSON file
# with open(output_json_path, "w", encoding="utf-8") as json_file:
#     json.dump(complete_output_data, json_file, indent=4, ensure_ascii=False)

# print(f"Successfully saved all transcription and diarization results to: {output_json_path}")